In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql import functions as F
import time
from pyspark.sql.functions import col, when, to_date, year, month, dayofmonth, trim, lower, round as spark_round,avg,count
import matplotlib.pyplot as plt
import numpy as np

In [2]:
import time
import os
import shutil
from pyspark.sql import SparkSession

# ============================================
# SETUP
# ============================================
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("CSV vs Parquet") \
    .master("local[4]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.default.parallelism", "8") \
    .getOrCreate()

# Verify configuration
print(f"Driver Memory: {spark.conf.get('spark.driver.memory')}")
print(f"Shuffle Partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Master URL: {spark.conf.get('spark.master')}")


Driver Memory: 8g
Shuffle Partitions: 8
Master URL: local[4]


In [3]:
df = spark.read.parquet("data/clean/")
df.cache()

DataFrame[Flow Duration: int, Total Fwd Packets: int, Total Backward Packets: int, Total Length of Fwd Packets: int, Total Length of Bwd Packets: int, Fwd Packet Length Max: int, Fwd Packet Length Min: int, Fwd Packet Length Mean: double, Fwd Packet Length Std: double, Bwd Packet Length Max: int, Bwd Packet Length Min: int, Bwd Packet Length Mean: double, Bwd Packet Length Std: double, Flow Bytes/s: double, Flow Packets/s: double, Flow IAT Mean: double, Flow IAT Std: double, Flow IAT Max: int, Flow IAT Min: int, Fwd IAT Total: int, Fwd IAT Mean: double, Fwd IAT Std: double, Fwd IAT Max: int, Fwd IAT Min: int, Bwd IAT Total: int, Bwd IAT Mean: double, Bwd IAT Std: double, Bwd IAT Max: int, Bwd IAT Min: int, Fwd Header Length34: bigint, Bwd Header Length: int, Fwd Packets/s: double, Bwd Packets/s: double, Min Packet Length: int, Max Packet Length: int, Packet Length Mean: double, Packet Length Std: double, Packet Length Variance: double, FIN Flag Count: int, SYN Flag Count: int, RST Flag

In [4]:
string_cols = [c for c, t in df.dtypes if t == "string"]
print(string_cols)

['Label']


In [5]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(
    inputCol="Label",
    outputCol="label_index",
    handleInvalid="keep"
)

df1 = indexer.fit(df).transform(df)

df1.select("Label", "label_index").show(10)

+------+-----------+
| Label|label_index|
+------+-----------+
|Normal|        0.0|
|Normal|        0.0|
|Normal|        0.0|
|Normal|        0.0|
|Normal|        0.0|
|Normal|        0.0|
|Normal|        0.0|
|Normal|        0.0|
|Normal|        0.0|
|Normal|        0.0|
+------+-----------+
only showing top 10 rows



In [7]:
# --- VectorAssembler ---
from pyspark.ml.feature import VectorAssembler
# Exclude the target label and any string columns from features
exclude_cols = set(string_cols + ["label_index"])
feature_cols = [c for c in df1.columns if c not in exclude_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw",
    handleInvalid="keep"   # skips rows with null/NaN in any feature col
)
df2 = assembler.transform(df1)


In [8]:
df2.select("Label", "label_index", "features_raw").show(10, truncate=True)

+------+-----------+--------------------+
| Label|label_index|        features_raw|
+------+-----------+--------------------+
|Normal|        0.0|[1092.0,9.0,6.0,3...|
|Normal|        0.0|[579225.0,132.0,1...|
|Normal|        0.0|[2.4368732E7,11.0...|
|Normal|        0.0|(58,[0,1,2,3,4,5,...|
|Normal|        0.0|[2941634.0,47.0,4...|
|Normal|        0.0|(58,[0,1,3,5,6,7,...|
|Normal|        0.0|[3785483.0,20.0,2...|
|Normal|        0.0|[312696.0,12.0,14...|
|Normal|        0.0|[791.0,9.0,4.0,63...|
|Normal|        0.0|(58,[0,1,3,5,6,7,...|
+------+-----------+--------------------+
only showing top 10 rows



In [9]:
# --- StandardScaler ---
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,   # zero-centre (dense output)
    withStd=True
)
scaler_model = scaler.fit(df2)
df3 = scaler_model.transform(df2)
# Show scaled features alongside label
df3.select("Label", "label_index", "features").show(10, truncate=True)

+------+-----------+--------------------+
| Label|label_index|            features|
+------+-----------+--------------------+
|Normal|        0.0|[-0.4707049050477...|
|Normal|        0.0|[-0.4542919837693...|
|Normal|        0.0|[0.22108082083110...|
|Normal|        0.0|[-0.4707310518002...|
|Normal|        0.0|[-0.3872243143156...|
|Normal|        0.0|[-0.4707358212404...|
|Normal|        0.0|[-0.3632678420024...|
|Normal|        0.0|[-0.4618586154879...|
|Normal|        0.0|[-0.4707134502947...|
|Normal|        0.0|[-0.4707357928509...|
+------+-----------+--------------------+
only showing top 10 rows



In [10]:
from pyspark.ml.stat import Summarizer
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Compute std for each numeric column individually
from pyspark.sql import functions as F

# Filter out columns with zero std or any nulls/NaN
good_cols = []
for c in feature_cols:
    stats = df1.select(
        F.stddev(F.col(c)).alias("std"),
        F.count(F.when(F.col(c).isNull() | F.isnan(F.col(c)), 1)).alias("null_count")
    ).first()
    if stats["std"] is not None and stats["std"] > 0 and stats["null_count"] == 0:
        good_cols.append(c)

print(f"Kept {len(good_cols)} / {len(feature_cols)} columns after variance/null check")

Kept 56 / 58 columns after variance/null check


In [11]:
assembler = VectorAssembler(
    inputCols=good_cols,
    outputCol="features_raw",
    handleInvalid="skip"    # drop any remaining bad rows
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

df2 = assembler.transform(df1)
scaler_model = scaler.fit(df2)
df3 = scaler_model.transform(df2)

# Update feature_cols to reflect what's actually in the vector
feature_cols = good_cols

In [12]:
from pyspark.ml.classification import RandomForestClassifier
import pandas as pd

rf_selector = RandomForestClassifier(
    featuresCol="features",
    labelCol="label_index",
    numTrees=100,
    maxDepth=10,
    seed=42
)

rf_model = rf_selector.fit(df3)

# Extract importances
importance_df = pd.DataFrame({
    "feature": good_cols,   # use good_cols, not feature_cols
    "importance": rf_model.featureImportances.toArray()
}).sort_values("importance", ascending=False).reset_index(drop=True)

print(importance_df.head(20))
print(f"\nTotal features: {len(good_cols)}")

                        feature  importance
0         Bwd Packet Length Std    0.101404
1             Packet Length Std    0.079467
2        Bwd Packet Length Mean    0.074164
3          Avg Bwd Segment Size    0.069996
4         Bwd Packet Length Max    0.063002
5           Average Packet Size    0.057218
6        Packet Length Variance    0.054911
7             Total Fwd Packets    0.032572
8   Total Length of Bwd Packets    0.030034
9   Total Length of Fwd Packets    0.028503
10             act_data_pkt_fwd    0.026208
11           Packet Length Mean    0.024064
12            Max Packet Length    0.022472
13         min_seg_size_forward    0.022030
14          Fwd Header Length34    0.019632
15          Fwd Header Length55    0.018339
16                    Idle Mean    0.015422
17        Fwd Packet Length Max    0.015294
18               ACK Flag Count    0.014728
19                 Fwd IAT Mean    0.014193

Total features: 56


In [13]:
# Cumulative 95% importance
importance_df["cumulative"] = importance_df["importance"].cumsum()
selected_features = importance_df[importance_df["cumulative"] <= 0.95]["feature"].tolist()

print(f"Selected {len(selected_features)} / {len(good_cols)} features")
print(importance_df[["feature", "importance", "cumulative"]])

Selected 35 / 56 features
                        feature    importance  cumulative
0         Bwd Packet Length Std  1.014045e-01    0.101404
1             Packet Length Std  7.946653e-02    0.180871
2        Bwd Packet Length Mean  7.416356e-02    0.255035
3          Avg Bwd Segment Size  6.999590e-02    0.325030
4         Bwd Packet Length Max  6.300194e-02    0.388032
5           Average Packet Size  5.721825e-02    0.445251
6        Packet Length Variance  5.491074e-02    0.500161
7             Total Fwd Packets  3.257185e-02    0.532733
8   Total Length of Bwd Packets  3.003392e-02    0.562767
9   Total Length of Fwd Packets  2.850300e-02    0.591270
10             act_data_pkt_fwd  2.620780e-02    0.617478
11           Packet Length Mean  2.406364e-02    0.641542
12            Max Packet Length  2.247176e-02    0.664013
13         min_seg_size_forward  2.202989e-02    0.686043
14          Fwd Header Length34  1.963207e-02    0.705675
15          Fwd Header Length55  1.833853e-02 

In [14]:
assembler_sel = VectorAssembler(
    inputCols=selected_features,
    outputCol="features_raw_sel",
    handleInvalid="skip"
)

scaler_sel = StandardScaler(
    inputCol="features_raw_sel",
    outputCol="features_selected",
    withMean=True,
    withStd=True
)

df2_sel = assembler_sel.transform(df1)
df_sel = scaler_sel.fit(df2_sel).transform(df2_sel)

print(f"Final: {df_sel.count()} rows x {len(selected_features)} features")

Final: 2522362 rows x 35 features


In [15]:
df_sel.limit(5).toPandas().head()

,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,...,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,label_index,features_raw_sel,features_selected
0,1092,9,6,3150,3152,1575,0,350.000000,694.509719,1576,...,0,0,0.0,0.0,0,0,Normal,0.0,"[813.8429005, 704.585067, 525.3333333, 525.333...","[0.4967251336276392, 0.5669242091779436, 0.292..."
1,579225,132,150,160,320799,160,0,1.212121,13.926212,4344,...,0,0,0.0,0.0,0,0,Normal,0.0,"[831.8348267, 1227.920672, 2138.66, 2138.66, 4...","[0.5171512065357831, 1.3589756764652234, 2.843..."
2,24368732,11,6,686,560,168,0,62.363636,80.651439,172,...,0,0,0.0,0.0,0,0,Normal,0.0,"[77.75517132, 78.24287643, 93.33333333, 93.333...","[-0.3389486822765582, -0.38102438504181635, -0..."
3,171,1,1,6,6,6,6,6.000000,0.000000,6,...,0,0,0.0,0.0,0,0,Normal,0.0,"(0.0, 0.0, 6.0, 6.0, 6.0, 9.0, 0.0, 1.0, 6.0, ...","[-0.4272234328903938, -0.4994424412357167, -0...."
4,2941634,47,46,2664,6954,456,0,56.680851,104.767466,976,...,0,0,0.0,0.0,0,0,Normal,0.0,"[307.3801341, 231.2838508, 151.173913, 151.173...","[-0.07825751563126083, -0.14940183319034522, -..."


In [16]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[
    indexer,          # StringIndexer
    assembler_sel,    # VectorAssembler (40 cols)
    scaler_sel        # StandardScaler
])

pipeline_model = pipeline.fit(df)
final_df = pipeline_model.transform(df).select("features_selected", "label_index")

In [17]:
final_df.printSchema()
final_df.show(5, truncate=True)
print("Total rows:", final_df.count())
print("Feature vector size:", len(final_df.head(1)[0]["features_selected"]))

root
 |-- features_selected: vector (nullable = true)
 |-- label_index: double (nullable = false)

+--------------------+-----------+
|   features_selected|label_index|
+--------------------+-----------+
|[0.49672513362763...|        0.0|
|[0.51715120653578...|        0.0|
|[-0.3389486822765...|        0.0|
|[-0.4272234328903...|        0.0|
|[-0.0782575156312...|        0.0|
+--------------------+-----------+
only showing top 5 rows

Total rows: 2522362
Feature vector size: 35


In [18]:
final_df.write.parquet("data/features/", mode="overwrite")

In [19]:
spark.stop()